# Train the waste sorting classifier on a Kaggle GPU

Runs the project's own training script, unchanged, from the public repository.

**Before running:** in the notebook's right-hand panel, set **Accelerator → GPU** (T4 or P100) and turn **Internet on**. Both need a phone-verified Kaggle account.

**After running:** download `run.zip` from the **Output** panel and unzip it into the project's `artifacts/` folder.

In [ ]:
# 1. A GPU must be available; otherwise stop here rather than train for hours on a CPU.
import torch

assert torch.cuda.is_available(), "No GPU: set Accelerator -> GPU in the right-hand panel"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. The code, from GitHub. Cloned to /tmp so the dataset is not saved as notebook output.
REPO = "https://github.com/vladzpop11/waste-sorting-classifier.git"
!rm -rf /tmp/repo && git clone -q --depth 1 {REPO} /tmp/repo
%cd /tmp/repo
!git log -1 --format='commit %H (%s)'

In [ ]:
# 3. Kaggle already has PyTorch, torchvision, scikit-learn, pandas, Pillow and PyYAML.
!pip install -q "mlflow>=3"

In [ ]:
# 4. Download TrashNet and check it: archive SHA-256, published class counts, then the
#    training script checks every image against the committed split.
!PYTHONPATH=src python -m waste_sorting_classifier.data

In [ ]:
# 5. Train (early stopping on validation loss). Outputs go to /kaggle/working/run.
!PYTHONPATH=src python -m waste_sorting_classifier.train --output-dir /kaggle/working/run

In [ ]:
# 6. One file to download: run.zip (model.pt, metrics, history, confusion matrix, MLflow store).
!git -C /tmp/repo rev-parse HEAD > /kaggle/working/run/commit.txt
!cd /kaggle/working && rm -f run.zip && zip -qr run.zip run && ls -la run run.zip